# Transform every channel in one imaging round

Registration is usually estimated from one structural channel. This notebook reuses that round's optional crop geometry, affine transform, and nonlinear warp for the remaining intensity channels. Every reused artifact must retain its matching readable `.complete.json` record, and all selected records must form one provenance chain.

In [ ]:
# EDIT THESE PATHS and naming rules before running the notebook.
from pathlib import Path

DATA_DIRECTORY = Path("path/to/unregistered_channels")
TRANSFORM_DIRECTORY = Path("path/to/registration_output")
OUTPUT_DIRECTORY = Path("path/to/registered_channels")
FIXED_IMAGE = Path("path/to/original_fixed_registration_channel.tif")
ROUND_NUMBER = 2
FIXED_CROP_INDICES: Path | None = (
    TRANSFORM_DIRECTORY / f"fixed_crop_indices_{ROUND_NUMBER}.json"
)
MOVING_CROP_INDICES: Path | None = (
    TRANSFORM_DIRECTORY / f"moving_crop_indices_{ROUND_NUMBER}.json"
)
# Set both crop paths to None when registration used crop=False.
CHANNEL_PATTERN = f"round{ROUND_NUMBER}_*.tif"
PIXEL_SIZE_XYZ = (0.15, 0.15, 0.15)
ITK_THREADS = 1
BLAS_THREADS = 1

For a `crop=True` registration, keep both crop paths set. The exclusive NumPy `(z0, z1, y0, y1, x0, x1)` bounds are rebuilt as physical ANTs images with the same origin calculation used during registration. For `crop=False`, set both paths to `None`. Before reading bounds or loading images, the notebook verifies exact artifact identity, stage, axis convention, common registration inputs, affine-to-nonlinear upstream identity, fixed geometry, round, crop policy and roles, configured XYZ pixel spacing, and the current fixed-image identity. The original moving registration channel need not remain available when transforms are applied to other channels.

In [ ]:
import importlib
import json
import math
from numbers import Real

from cyexm.io.images import (
    ants_to_external_array,
    crop_array,
    external_to_ants_array,
    load_ants_image,
)
from cyexm.io.tiff import write_uint16_tiff
from cyexm.registration.artifacts import ArtifactStore, CompletionRecord, FileIdentity
from cyexm.registration.transforms import transform_image
from cyexm.runtime import NativeThreadConfig, configure_native_threads
from cyexm.spatial import AXIS_CONVENTION_VERSION

NATIVE_THREADS = NativeThreadConfig(ITK_THREADS, BLAS_THREADS)
configure_native_threads(NATIVE_THREADS)
ants = importlib.import_module("ants")

AFFINE_TRANSFORM = TRANSFORM_DIRECTORY / f"linear_transform_{ROUND_NUMBER}.mat"
NONLINEAR_WARP = TRANSFORM_DIRECTORY / f"nonlinear_warp_{ROUND_NUMBER}.nii.gz"


def validate_cached_artifact(path, expected_stage):
    if path is None:
        return None
    record_path = ArtifactStore().record_path(path)
    try:
        record = CompletionRecord.read(record_path)
    except Exception as error:
        raise ValueError(
            f"Artifact completion record is missing or unreadable: {record_path}"
        ) from error
    try:
        identity = FileIdentity.from_path(path)
    except Exception as error:
        raise ValueError(f"Artifact identity could not be read: {path}") from error
    if record.artifact != identity:
        raise ValueError(
            f"Artifact identity does not match its completion record: {path}"
        )
    if record.request.stage != expected_stage:
        raise ValueError(
            f"Artifact stage {record.request.stage!r} is not {expected_stage!r}: {path}"
        )
    if (
        record.request.parameters.get("axis_convention_version")
        != AXIS_CONVENTION_VERSION
    ):
        raise ValueError(f"Artifact axis convention is incompatible: {path}")
    return record


def normalize_pixel_size_xyz(value):
    message = "Pixel spacing must contain exactly three positive finite numbers"
    if isinstance(value, (str, bytes)):
        raise ValueError(message)
    try:
        values = tuple(value)
    except TypeError as error:
        raise ValueError(message) from error
    if len(values) != 3:
        raise ValueError(message)
    normalized = []
    for item in values:
        if isinstance(item, bool) or not isinstance(item, Real):
            raise ValueError(message)
        try:
            number = float(item)
        except (OverflowError, TypeError, ValueError) as error:
            raise ValueError(message) from error
        if not math.isfinite(number) or number <= 0.0:
            raise ValueError(message)
        normalized.append(number)
    return tuple(normalized)


def validate_registration_chain(
    affine_record,
    nonlinear_record,
    fixed_crop_record,
    moving_crop_record,
    fixed_image,
    round_number,
    pixel_size_xyz,
):
    if affine_record is None or nonlinear_record is None:
        raise ValueError(
            "Registration chain inputs require affine and nonlinear records"
        )
    if isinstance(round_number, bool) or not isinstance(round_number, int):
        raise ValueError("Registration chain round number must be an integer")
    if round_number < 0:
        raise ValueError("Registration chain round number must be non-negative")
    crop_selected = fixed_crop_record is not None
    if crop_selected != (moving_crop_record is not None):
        raise ValueError(
            "Registration chain crop policy requires both crop records or neither"
        )
    if affine_record.request.upstream is not None:
        raise ValueError("Registration chain affine upstream must be absent")
    if nonlinear_record.request.upstream != affine_record.artifact:
        raise ValueError(
            "Registration chain nonlinear upstream does not match the affine artifact"
        )
    inputs = affine_record.request.inputs
    if len(inputs) != 2 or len(nonlinear_record.request.inputs) != 2:
        raise ValueError("Registration chain inputs must contain exactly two files")
    if nonlinear_record.request.inputs != inputs:
        raise ValueError("Registration chain inputs do not match")
    if nonlinear_record.request.geometry != affine_record.request.geometry:
        raise ValueError("Registration chain fixed geometry does not match")
    crop_members = (
        (fixed_crop_record, "crop:fixed", "fixed"),
        (moving_crop_record, "crop:moving", "moving"),
    )
    for record, expected_stage, expected_role in crop_members:
        if record is None:
            continue
        if record.request.stage != expected_stage:
            raise ValueError(
                f"Registration chain {expected_role} crop stage does not match"
            )
        if record.request.parameters.get("image_role") != expected_role:
            raise ValueError(
                f"Registration chain {expected_role} crop role does not match"
            )
        if record.request.upstream is not None:
            raise ValueError(
                f"Registration chain {expected_role} crop upstream must be absent"
            )
        if record.request.inputs != inputs:
            raise ValueError(
                f"Registration chain {expected_role} crop inputs do not match"
            )
    records = [affine_record, nonlinear_record]
    records.extend(record for record, _, _ in crop_members if record is not None)
    try:
        expected_pixel_size = normalize_pixel_size_xyz(pixel_size_xyz)
    except ValueError as error:
        raise ValueError(
            "Registration chain pixel spacing configuration is invalid"
        ) from error
    for record in records:
        recorded_round = record.request.parameters.get("imaging_round")
        if (
            isinstance(recorded_round, bool)
            or not isinstance(recorded_round, int)
            or recorded_round != round_number
        ):
            raise ValueError("Registration chain round does not match")
        recorded_crop = record.request.parameters.get("crop")
        if not isinstance(recorded_crop, bool) or recorded_crop is not crop_selected:
            raise ValueError("Registration chain crop policy does not match")
        try:
            recorded_pixel_size = normalize_pixel_size_xyz(
                record.request.parameters.get("pixel_size_xyz")
            )
        except ValueError as error:
            raise ValueError(
                "Registration chain pixel spacing record is invalid"
            ) from error
        if recorded_pixel_size != expected_pixel_size:
            raise ValueError("Registration chain pixel spacing does not match")
    try:
        fixed_identity = FileIdentity.from_path(fixed_image)
    except Exception as error:
        raise ValueError(
            f"Registration chain fixed image identity could not be read: {fixed_image}"
        ) from error
    if fixed_identity != inputs[0]:
        raise ValueError(
            "Registration chain fixed image does not match the recorded fixed input"
        )


def read_bounds(path):
    return None if path is None else tuple(json.loads(path.read_text(encoding="utf-8")))


affine_record = validate_cached_artifact(AFFINE_TRANSFORM, "affine")
nonlinear_record = validate_cached_artifact(NONLINEAR_WARP, "nonlinear")
fixed_crop_record = validate_cached_artifact(FIXED_CROP_INDICES, "crop:fixed")
moving_crop_record = validate_cached_artifact(MOVING_CROP_INDICES, "crop:moving")
validate_registration_chain(
    affine_record,
    nonlinear_record,
    fixed_crop_record,
    moving_crop_record,
    FIXED_IMAGE,
    ROUND_NUMBER,
    PIXEL_SIZE_XYZ,
)
fixed_bounds = read_bounds(FIXED_CROP_INDICES)
moving_bounds = read_bounds(MOVING_CROP_INDICES)


def crop_spatial_image(image, bounds):
    if bounds is None:
        return image
    z0, _z1, y0, _y1, x0, _x1 = bounds
    data = crop_array(ants_to_external_array(image.numpy()), bounds)
    origin = ants.transform_index_to_physical_point(image, (x0, y0, z0))
    origin = tuple(float(value) for value in origin)
    return ants.from_numpy(
        external_to_ants_array(data),
        origin=origin,
        spacing=image.spacing,
        direction=image.direction,
    )


fixed_full = load_ants_image(FIXED_IMAGE, PIXEL_SIZE_XYZ)
fixed = crop_spatial_image(fixed_full, fixed_bounds)
channel_paths = sorted(DATA_DIRECTORY.glob(CHANNEL_PATTERN))
if not channel_paths:
    raise FileNotFoundError(
        f"No channels match {CHANNEL_PATTERN!r} in {DATA_DIRECTORY}"
    )
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

In [ ]:
for channel_path in channel_paths:
    moving_full = load_ants_image(channel_path, PIXEL_SIZE_XYZ)
    moving = crop_spatial_image(moving_full, moving_bounds)
    registered = transform_image(
        fixed=fixed,
        moving=moving,
        affine_path=AFFINE_TRANSFORM,
        warp_path=NONLINEAR_WARP,
        interpolation="linear",
        native_threads=NATIVE_THREADS,
    )
    output_path = OUTPUT_DIRECTORY / channel_path.name
    write_uint16_tiff(
        registered,
        output_path,
        float(moving.numpy().min()),
        float(moving.numpy().max()),
    )
    print(f"Wrote {output_path}")